# Part I – Reviews Sentiment Classification using TF-IDF


In [ ]:
import nltk
import re
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
 
#required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') 

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/applecare/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/applecare/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/applecare/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Load Data

In [ ]:
data = pd.read_csv('amazon_reviews.csv')
data = data.dropna(subset=['cleaned_review', 'sentiments'])
data['cleaned_review'] = data['cleaned_review'].astype(str)

# Step 1: pre-processing

In [ ]:

stop_words = set(stopwords.words('english'))
 
#keep negation words so sentiment meaning is preserved
words_to_keep = {'not', 'no', 'very'}
custom_stop_words = stop_words - words_to_keep

def preprocess_text(review: str) -> str:
    """Lowercase, expand contractions, remove punctuation,
    tokenize, and strip stop words."""
 
    #1.lowercase
    review = review.lower()
 
    #2.expand common contractions before removing punctuation
    review = review.replace("n't", " not")
    review = review.replace("'re", " are")
    review = review.replace("'ve", " have")
    review = review.replace("'ll", " will")
    review = review.replace("'d",  " would")
    review = review.replace("'m",  " am")
 
    #3.remove punctuation / special characters
    review = re.sub(r'[^\w\s]', '', review)
 
    # 4. Tokenize
    words = word_tokenize(review)
 
    # 5. Remove stop words and non-alpha tokens
    clean_words = [
        word for word in words
        if word not in custom_stop_words and word.isalpha()
    ]
    return " ".join(clean_words)

data['processed_review'] = data['cleaned_review'].apply(preprocess_text)
#drop any rows where preprocessing produced an empty string
data = data[data['processed_review'].str.strip() != '']
data.reset_index(drop=True, inplace=True)

# Step 2: label encoding


In [ ]:
le = LabelEncoder()
data['label_encoded'] = le.fit_transform(data['sentiments'])
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Label mapping: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


# Step 3: Data Splitting

In [ ]:
X = data['processed_review']
y = data['label_encoded']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keeps class proportions equal in both splits
)

# Step 4: TF-IDF VECTORIZATION

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    #apply log normalization to term frequencies
    stop_words=None,      #stop words already removed in preprocessing
)
 
X_train_tfidf = vectorizer.fit_transform(X_train)   #fit+transform on train
X_test_tfidf  = vectorizer.transform(X_test)         #transform only on test

# Step 5: Text classification

In [7]:
# 5.1 Logistic Regression

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_tfidf, y_train)
y_pred_lr = logistic_model.predict(X_test_tfidf)


In [8]:
# 5.2: SVM

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)

# Step 6 – Classification Reports

In [ ]:
target_names = le.classes_.tolist()   #['Negative', 'Neutral', 'Positive']
 
print("\n" + "=" * 50)
print("          Logistic Regression Results")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=target_names))
 
print("\n" + "=" * 50)
print("                  SVM Results")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_svm):.4f}")
print(classification_report(y_test, y_pred_svm, target_names=target_names))
 
#models comparison
lr_acc  = accuracy_score(y_test, y_pred_lr)
svm_acc = accuracy_score(y_test, y_pred_svm)
 
print("\n" + "=" * 50)
print("             Model Comparison")
print("=" * 50)
print(f"Logistic Regression Accuracy : {lr_acc:.4f}")
print(f"SVM Accuracy                 : {svm_acc:.4f}")
 
if lr_acc > svm_acc:
    print(">> Logistic Regression performed better overall.")
elif svm_acc > lr_acc:
    print(">> SVM performed better overall.")
else:
    print(">> Both models performed equally.")



          Logistic Regression Results
Accuracy : 0.8366
              precision    recall  f1-score   support

    negative       0.84      0.38      0.52       307
     neutral       0.77      0.83      0.80      1256
    positive       0.88      0.92      0.90      1900

    accuracy                           0.84      3463
   macro avg       0.83      0.71      0.74      3463
weighted avg       0.84      0.84      0.83      3463


                  SVM Results
Accuracy : 0.8527
              precision    recall  f1-score   support

    negative       0.79      0.52      0.63       307
     neutral       0.78      0.85      0.81      1256
    positive       0.91      0.91      0.91      1900

    accuracy                           0.85      3463
   macro avg       0.83      0.76      0.78      3463
weighted avg       0.85      0.85      0.85      3463


             Model Comparison
Logistic Regression Accuracy : 0.8366
SVM Accuracy                 : 0.8527
>> SVM performed better o

# Step 7(Bonus): Predict the label of new review

In [ ]:
def predict_review(review_text: str, model) -> str:
    """Pre-process a raw review, vectorize it with the SAME
    TF-IDF vectorizer (transform only), and return the predicted label."""
    processed = preprocess_text(review_text)
    vector     = vectorizer.transform([processed])
    prediction = model.predict(vector)
    return le.inverse_transform(prediction)[0]

#example
user_input = input("\nEnter a review to classify: ")
 
lr_result  = predict_review(user_input, logistic_model)
svm_result = predict_review(user_input, svm_model)
 
print(f"\nLogistic Regression → {lr_result}")
print(f"SVM→ {svm_result}")

# Positive (2):"This product is amazing! The quality is excellent and it exceeded my expectations. I would definitely buy it again."

# Neutral (1): "The product is okay. It works as described but nothing special. Delivery was fast though."

# Negative (0): "The product stopped working after two days. Very poor quality and a complete waste of money. I am really disappointed." 
